# Retinal Vessel Mask Sensitivity Analysis

Controlled perturbation experiments on the 45-image HRF retinal vessel segmentation dataset.

In [ ]:
from pathlib import Path
import sys
import os
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

from retina_sensitivity.data import ensure_hrf_data, load_hrf_sample
from retina_sensitivity.experiment import run_experiment
from retina_sensitivity.perturbations import make_perturbations

DATA_ROOT = ensure_hrf_data(ROOT)
RESULTS = ROOT / "results"

## Inspect a real HRF image and its expert annotation

In [ ]:
sample = load_hrf_sample(DATA_ROOT, "01_h")
variants = make_perturbations(sample.vessel_mask, sample.fov_mask)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(sample.image)
axes[0].set_title("Original fundus image")
axes[1].imshow(sample.vessel_mask, cmap="gray")
axes[1].set_title("Expert vessel annotation")
axes[2].imshow(variants["erosion_r1"], cmap="gray")
axes[2].set_title("Controlled erosion (1 px)")
for ax in axes:
    ax.axis("off")
plt.tight_layout()

## Run the sensitivity experiment across the HRF dataset

In [ ]:
SAMPLE_LIMIT = int(os.environ.get("RETINA_SAMPLE_LIMIT", "0")) or None
per_image, summary = run_experiment(DATA_ROOT, RESULTS, sample_limit=SAMPLE_LIMIT)
summary

## Compare mask overlap and feature sensitivity

In [ ]:
display(pd.read_csv(RESULTS / "per_image_features.csv").head())
display(pd.read_csv(RESULTS / "summary.csv"))

from IPython.display import Image, display
for figure in ("mask_overlap.png", "feature_sensitivity.png"):
    display(Image(filename=str(RESULTS / "figures" / figure)))

## Reproducibility

All image-level records and summary tables are saved under `results/`. Perturbed masks are derived from the original expert annotations and are not additional patient data.